# Rename file train
Jalankan sel berurutan. Setiap file langsung di dalam `train/<kelas>/` menjadi `<kelas>_001`, `<kelas>_002`, dst. Ekstensi asli dipertahankan; indeks mulai dari 001 per kelas, berdasarkan urutan nama awal. Folder `test` tidak disentuh.

Tutup aplikasi yang sedang memakai dataset. Jangan hentikan kernel saat rename berlangsung. Jalankan notebook dari folder dataset, atau ubah `TRAIN_DIR` pada sel berikut.

In [1]:
from pathlib import Path
from uuid import uuid4

TRAIN_DIR = Path.cwd() / "train"
if not TRAIN_DIR.is_dir():
    raise FileNotFoundError(f"Folder train tidak ditemukan: {TRAIN_DIR}. Ubah TRAIN_DIR sesuai lokasi dataset.")


def rename_train(train_dir):
    train_dir = Path(train_dir)
    if not train_dir.is_dir():
        raise FileNotFoundError(train_dir)
    plan = []
    for folder in sorted(p for p in train_dir.iterdir() if p.is_dir()):
        files = sorted((p for p in folder.iterdir() if p.is_file()), key=lambda p: p.name)
        for index, source in enumerate(files, 1):
            target = folder / f"{folder.name}_{index:03d}{source.suffix}"
            if source != target:
                plan.append((source, target))
    sources = {source for source, _ in plan}
    for _, target in plan:
        if target.exists() and target not in sources:
            raise FileExistsError(f"Nama tujuan sudah dipakai: {target}")

    # Nama sementara memungkinkan pertukaran nama tanpa menimpa file.
    staged = []
    finished = []
    try:
        for source, target in plan:
            temporary = source.with_name(f".rename_{uuid4().hex}{source.suffix}")
            source.rename(temporary)
            staged.append((source, temporary, target))
        for source, temporary, target in staged:
            temporary.rename(target)
            finished.append((source, temporary, target))
    except Exception:
        for source, temporary, target in reversed(finished):
            target.rename(temporary)
        for source, temporary, target in reversed(staged):
            temporary.rename(source)
        raise
    return len(plan)


In [2]:
changed = rename_train(TRAIN_DIR)
print(f"Selesai: {changed} file diubah namanya di {TRAIN_DIR}")
for folder in sorted(p for p in TRAIN_DIR.iterdir() if p.is_dir()):
    print(f"{folder.name}: {sum(p.is_file() for p in folder.iterdir())} file")


Selesai: 2487 file diubah namanya di c:\Users\Ricardo Cuthbert\Downloads\skrpsi bwabwa\train
apa_kabar: 306 file
bingung: 306 file
halo: 325 file
kamu: 302 file
marah: 303 file
saya: 335 file
senang: 305 file
tidur: 305 file
